### Assignment 1

Construct a Gaussian mixture model class using only numpy

In [1]:
import numpy as np
import cv2

In [2]:
class GaussianMixture:
    def __init__(self, n_components, max_iter=100, tol=1e-6, random_state=None):
        self.K = n_components
        self.max_iter = max_iter
        self.tol = tol
        self.random_state = random_state
        self.log_likelihood = -np.inf # Initialize log_likelihood

    def _initialize_parameters(self, X):
        N, D = X.shape
        np.random.seed(self.random_state)
        # Randomly choose K data points as initial means
        self.mu = X[np.random.choice(N, self.K, replace=False)].astype(np.float64)
        # Initialize covariances as identity matrices
        self.sigma = np.array([np.eye(D) for _ in range(self.K)]).astype(np.float64)
        # Initialize mixing coefficients uniformly
        self.pi = np.full(self.K, 1 / self.K).astype(np.float64)

    def _multivariate_gaussian(self, X_n, mean_k, cov_k):
        # Calculate multivariate Gaussian probability density for given X_n, mean_k, and cov_k
        D = X_n.shape[-1]
        try:
            cov_det = np.linalg.det(cov_k)
            cov_inv = np.linalg.inv(cov_k)
        except np.linalg.LinAlgError:
            # Handle singular matrix by adding a small diagonal perturbation
            cov_k += 1e-6 * np.eye(D)
            cov_det = np.linalg.det(cov_k)
            cov_inv = np.linalg.inv(cov_k)

        # Ensure positive definite covariance
        if cov_det <= 0:
            cov_det = 1e-10 # Prevent log(0) or division by zero
        
        norm_const = 1. / np.sqrt((2 * np.pi) ** D * cov_det)
        
        # Vectorized calculation for multiple points (if X_n is (N_batch, D))
        if X_n.ndim == 2:
            diff = X_n - mean_k
            exponent = -0.5 * np.sum(diff @ cov_inv * diff, axis=1)
        else: # Single point (D,)
            diff = X_n - mean_k
            exponent = -0.5 * diff.T @ cov_inv @ diff
            
        return norm_const * np.exp(exponent)

    def _e_step(self, X):
        N = X.shape[0]
        self.gamma = np.zeros((N, self.K))
        for k in range(self.K):
            # Calculate P(x_n | C_k) for all n for component k
            self.gamma[:, k] = self.pi[k] * self._multivariate_gaussian(X, self.mu[k], self.sigma[k])
        
        # Normalize gamma for each pixel
        sum_gamma = np.sum(self.gamma, axis=1, keepdims=True)
        # Add a small epsilon to avoid division by zero
        self.gamma = self.gamma / (sum_gamma + 1e-10) 

    def _m_step(self, X):
        N, D = X.shape
        N_k = np.sum(self.gamma, axis=0) # Sum of responsibilities for each component

        # Update means
        self.mu = (self.gamma.T @ X) / (N_k[:, np.newaxis] + 1e-10)

        # Update covariances
        for k in range(self.K):
            diff = X - self.mu[k]
            # Weighted outer product sum, then normalize
            # Add regularization term to avoid singular matrices
            self.sigma[k] = (diff.T @ (self.gamma[:, k, np.newaxis] * diff)) / (N_k[k] + 1e-10)
            self.sigma[k] += 1e-6 * np.eye(D) # Regularization

        # Update mixing coefficients
        self.pi = N_k / (N + 1e-10)

    def _compute_log_likelihood(self, X):
        N = X.shape[0]
        log_likelihood = 0
        for n in range(N):
            prob = 0
            for k in range(self.K):
                prob += self.pi[k] * self._multivariate_gaussian(X[n], self.mu[k], self.sigma[k])
            # Add a small epsilon to prevent log(0)
            log_likelihood += np.log(prob + 1e-10)
        return log_likelihood

    def fit(self, X):
        self._initialize_parameters(X)
        log_likelihood_old = -np.inf # Initialize with a very small number

        for iteration in range(self.max_iter):
            self._e_step(X)
            self._m_step(X)
            log_likelihood = self._compute_log_likelihood(X)
            
            # Check for convergence
            if log_likelihood_old is not None and abs(log_likelihood - log_likelihood_old) < self.tol:
                print(f"GMM converged at iteration {iteration}")
                break
            log_likelihood_old = log_likelihood
        self.log_likelihood = log_likelihood

    def predict_proba(self, X):
        N = X.shape[0]
        probs = np.zeros((N, self.K))
        for k in range(self.K):
            probs[:, k] = self.pi[k] * self._multivariate_gaussian(X, self.mu[k], self.sigma[k])
        
        # Normalize probabilities for each pixel
        sum_probs = np.sum(probs, axis=1, keepdims=True)
        probs_normalized = probs / (sum_probs + 1e-10)
        return probs_normalized

    def predict(self, X):
        return np.argmax(self.predict_proba(X), axis=1)


In [3]:
import cv2

In [4]:
def segment_background(image_path, n_components=3):
    img = cv2.imread(image_path)
    if img is None:
        raise FileNotFoundError(f"Image not found at {image_path}")

    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w, c = img.shape

    pixels = img_rgb.reshape(-1, 3).astype(np.float64)

    gmm = GaussianMixture(n_components=n_components, max_iter=50, random_state=42)
    gmm.fit(pixels)

    labels = gmm.predict(pixels)

    background_label = np.bincount(labels).argmax()
    
    # Initial mask from GMM
    mask = (labels != background_label).astype(np.uint8) * 255
    mask = mask.reshape(h, w)

    # --- Start of new morphological operations ---
    # Define a kernel for morphological operations
    # A 3x3 or 5x5 elliptical/rectangular kernel is typically good.
    # Experiment with kernel size to get desired results.
    kernel_size = 5 # Example size
    kernel = np.ones((kernel_size, kernel_size), np.uint8) # Rectangular kernel

    # Perform opening: erosion followed by dilation
    # Erosion removes small specks (grass fragments)
    mask = cv2.erode(mask, kernel, iterations=1)
    # Dilation fills in small holes and restores object size
    mask = cv2.dilate(mask, kernel, iterations=1)
    # --- End of new morphological operations ---

    # Apply the refined mask
    img_bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
    foreground = cv2.bitwise_and(img_bgr, img_bgr, mask=mask)

    return foreground, mask

In [5]:
image_path = "cow.jpg"
foreground, mask = segment_background(image_path)

# Save the results
cv2.imwrite("foreground.jpg", foreground)
cv2.imwrite("mask.jpg", mask)

print("Segmentation complete. 'foreground.jpg' and 'mask.jpg' saved.")

Segmentation complete. 'foreground.jpg' and 'mask.jpg' saved.
